# LTX Bulk Renderer - Kaggle Launcher

This notebook is a thin orchestration layer for the repository.

It does **not** contain rendering, synchronization, validation, or business logic. Those responsibilities remain inside repository modules and are invoked through shared orchestration helpers.


## Notebook Flow

1. Bootstrap the repository import path.
2. Inspect the current environment and install only missing dependencies.
3. Prepare the launcher by discovering datasets, configuration, credentials, resume artifacts, diagnostics, and preflight state.
4. Execute the production pipeline with a live dashboard.
5. Review output artifacts and fail the notebook explicitly if the run was not successful.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from orchestration.kaggle import (
    KaggleNotebookLauncher,
    ensure_notebook_dependencies,
    inspect_runtime_dependencies,
)

print(f"Repository root: {REPO_ROOT}")


## Phase 1 - Dependency Validation

This phase reports the current environment first, then installs only the missing packages required by the repository launcher.


In [ ]:
def print_dependency_report(title, inspection):
    print(title)
    for package in inspection.packages:
        status = "installed" if package.installed else "missing"
        version = package.version or "n/a"
        print(f" - {package.package_name}: {status} ({version})")
    print(f"FFmpeg: {inspection.ffmpeg_version if inspection.ffmpeg_path else 'missing'}")
    print(f"CUDA available: {inspection.cuda_available}")
    print(f"CUDA version: {inspection.cuda_version}")
    print(f"GPU: {inspection.gpu_name}")

inspection_before = inspect_runtime_dependencies()
print_dependency_report("Dependency inspection before install", inspection_before)

inspection_after = ensure_notebook_dependencies()
print()
print_dependency_report("Dependency inspection after install", inspection_after)


## Phase 2 - Environment and Project Preparation

This phase validates the repository layout, discovers Kaggle inputs, detects Google Drive credentials, restores resume seeds when available, and runs diagnostics plus preflight using repository modules.


In [ ]:
launcher = KaggleNotebookLauncher(REPO_ROOT)
context = launcher.prepare()
launcher.display_preparation()


In [ ]:
execution_plan = {
    "jobs_csv_path": str(context.discovery.jobs_csv_path),
    "reference_images_dir": str(context.discovery.reference_images_dir),
    "project_config_path": str(context.discovery.project_config_path) if context.discovery.project_config_path else None,
    "output_dir": str(context.config.output_dir),
    "log_dir": str(context.config.log_dir),
    "manifest_path": str(context.config.manifest_path),
    "heartbeat_path": str(context.config.heartbeat_path),
    "report_path": str(context.config.report_path),
    "drive_mode": context.drive_credentials.source,
    "resume_manifest_present": bool(context.resume_summary.manifest_path),
    "cache_index_present": bool(context.resume_summary.cache_index_path),
    "preflight_ready": context.preparation.ready,
}

for key, value in execution_plan.items():
    print(f"{key}: {value}")


## Phase 3 - Execute Production Pipeline

This phase runs the repository pipeline and refreshes a live dashboard using repository heartbeat and manifest artifacts.


In [ ]:
result = launcher.run_with_dashboard(refresh_seconds=5)
result_payload = result.to_dict()
result_payload


## Phase 4 - Final Artifact Review

This phase shows the primary artifact locations and fails the notebook explicitly if the production run did not succeed.


In [ ]:
artifact_paths = {
    "report_json": context.config.report_path,
    "summary_txt": context.config.summary_path,
    "diagnostics_json": context.config.diagnostics_path,
    "validation_report_json": context.config.validation_report_path,
    "performance_json": context.config.performance_report_path,
    "benchmark_json": context.config.benchmark_json_path,
    "manifest_json": context.config.manifest_path,
    "heartbeat_json": context.config.heartbeat_path,
    "stitched_output": context.config.stitched_output_path,
    "log_dir": context.config.log_dir,
}

for name, path in artifact_paths.items():
    exists = path.exists() if hasattr(path, "exists") else False
    print(f"{name}: {path} | exists={exists}")

if not result.success:
    raise RuntimeError(result.failure.get("summary", "Pipeline execution failed"))
